In [1]:
from pydantic import BaseModel, Field
from typing import Literal, Union
import os
import logging
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO)

In [2]:
os.getenv("HF_HOME")
os.getenv("GOOGLE_API_KEY")
os.getenv("GROQ_API_KEY")
os.getenv("NVIDIA_API_KEY")
os.getenv("NVIDIA_API_KEY")
os.getenv("OPENROUTER_API_KEY")
os.getenv("ITI_API_KEY")
print()

In [3]:
class PrinciplesMetadata(BaseModel):

    topic: list[
        Literal[
            "program_design",
            "periodization",
            "progressive_overload",
            "volume",
            "frequency",
            "intensity",
            "load",
            "exercise_selection",
            "recovery",
            "fatigue",
            "warmup",
            "energy_systems",
        ]
    ] = Field(min_length=1)

    planner_stage: list[
        Literal[
            "goal_selection",
            "program_design",
            "exercise_selection",
            "progression",
            "recovery",
        ]
    ] = Field(min_length=1)

    goals: list[
        Literal[
            "hypertrophy",
            "strength",
            "fat_loss",
            "endurance",
        ]
    ] = Field(min_length=1)

    applies_to: list[
        Literal[
            "all",
            "hypertrophy",
            "strength",
            "fat_loss",
            #"endurance",
        ]
    ] = Field(min_length=1)

    knowledge_type: list[
        Literal[
            "definition",
            "principle",
            "recommendation",
            "warning",
            "protocol",
        ]
    ] = Field(min_length=1)

In [4]:
class GoalNamespaceMetadata(BaseModel):

    muscle: list[
        Literal[
            "all",
            "chest",
            "back",
            "shoulders",
            "biceps",
            "triceps",
            "forearms",
            "quads",
            "hamstrings",
            "glutes",
            "calves",
            "abs",
        ]
    ] = Field(min_length=1)

    topic: list[
        Literal[
            # Science
            "muscle_physiology",
            "neuromuscular_system",
            "muscle_activation",
            "biomechanics",

            # Goal-specific
            "muscle_growth_mechanisms",
            "hypertrophy_programming",
            "maximal_strength",
            "force_production",
            "power_development",
            "neural_adaptation",

            # Programming
            "volume",
            "frequency",
            "intensity",
            "load",
            "exercise_selection",
            "exercise_order",
            "periodization",
            "recovery",
            "fatigue_management",
            "advanced_techniques",
        ]
    ] = Field(min_length=1)

    experience_level: Literal[
        "all",
        "beginner",
        "intermediate",
        "advanced",
    ] = Field(min_length=1)

    goals: list[
        Literal[
            "hypertrophy",
            "strength",
            "fat_loss",
        ]
    ] = Field(min_length=1)

In [5]:
class CollectionRoute(BaseModel):
    collection: Literal[
        "principles",
        "hypertrophy",
        "strength"
    ]

In [6]:
class Section(BaseModel):
    title: str = Field(description="Current section title")
    level: int = Field(description="Markdown heading level (1-6)")
    parent_titles: list[str] = Field(default_factory=list)
    content: str = Field(description="Section content")

In [7]:
from pathlib import Path
def save_markdown(markdown: str, output_path: Path) -> None:

    output_path = Path(output_path)

    output_path.write_text(
        markdown,
        encoding="utf-8",
    )

In [8]:
from dataclasses import dataclass

@dataclass
class ChapterInfo:
    title: str
    start_page: int
    end_page: int


@dataclass
class BookInfo:
    title: str
    author: str
    slug: str

@dataclass
class BookPaths:
    book_dir: Path
    chapters_dir: Path
    markdown_dir: Path
    chunks_dir: Path

In [9]:
import re

def slugify(text: str) -> str:
    text = text.lower().strip()

    text = re.sub(r"[^\w\s-]", "", text)

    text = re.sub(r"[-\s]+", "_", text)

    return text

In [10]:
import fitz
book_path = "books/Science-and-development-of-muscle-hypertrophy-by-Brad-Schoenfeld-z-lib.org_.pdf"


def extract_book_info(pdf_path: str) -> BookInfo:
    doc = fitz.open(pdf_path)

    metadata = doc.metadata

    title = metadata.get("title", "").strip()
    author = metadata.get("author", "").strip().rstrip(";")

    doc.close()
    return BookInfo(
        title=title,
        author=author,
        slug=slugify(title),
    )


In [11]:
book = extract_book_info(book_path)

print(book)

BookInfo(title='Science and Development of Muscle Hypertrophy', author='Brad Schoenfeld', slug='science_and_development_of_muscle_hypertrophy')


In [12]:
def create_book_folders(
    root_dir: Path,
    book: BookInfo,
) -> dict[str, Path]:
    """
    Create the folder structure for a book.

    Structure:
    data/
    └── books/
        └── <book_slug>/
            ├── chapters/
            └── markdown/
    """

    book_dir = root_dir / book.slug

    chapters_dir = book_dir / "chapters"
    markdown_dir = book_dir / "markdown"
    chunks_dir = book_dir / "chunks"


    chapters_dir.mkdir(parents=True, exist_ok=True)
    markdown_dir.mkdir(parents=True, exist_ok=True)
    chunks_dir.mkdir(parents=True, exist_ok=True)

    return BookPaths(
        book_dir=book_dir,
        chapters_dir=chapters_dir,
        markdown_dir=markdown_dir,
        chunks_dir=chunks_dir,
    )

In [13]:
paths = create_book_folders(
    root_dir=Path("data/books"),
    book=book,
)
print(paths.chapters_dir)
print(paths.markdown_dir)
print(paths.chunks_dir)

data/books/science_and_development_of_muscle_hypertrophy/chapters
data/books/science_and_development_of_muscle_hypertrophy/markdown
data/books/science_and_development_of_muscle_hypertrophy/chunks


In [14]:
def extract_chapters(pdf_path: Path) -> list[ChapterInfo]:
    doc = fitz.open(pdf_path)
    toc = doc.get_toc()

    chapters = []

    for i, (_, title, start_page) in enumerate(toc):
        if not title.lower().startswith("chapter"):
            continue

        if i + 1 < len(toc):
            next_page = toc[i + 1][2]
            end_page = next_page - 1
        else:
            end_page = doc.page_count

        chapters.append(
            ChapterInfo(
                title=title,
                start_page=start_page,
                end_page=end_page,
            )
        )

    doc.close()

    return chapters

In [15]:
chapter_infos = extract_chapters(book_path)

for chapter in chapter_infos:
    print(chapter)

ChapterInfo(title='Chapter 1: Hypertrophy-Related Responses and Adaptations to Exercise Stress', start_page=10, end_page=38)
ChapterInfo(title='Chapter 2: Mechanisms of Hypertrophy', start_page=39, end_page=65)
ChapterInfo(title='Chapter 3: The Measurement of Muscle Hypertrophy', start_page=66, end_page=86)
ChapterInfo(title='Chapter 4: Role of Resistance Training Variables in Hypertrophy', start_page=87, end_page=144)
ChapterInfo(title='Chapter 5: Advanced Training Practices', start_page=145, end_page=158)
ChapterInfo(title='Chapter 6: Role of Aerobic Training in Hypertrophy', start_page=159, end_page=174)
ChapterInfo(title='Chapter 7: Factors in Maximal Hypertrophic Development', start_page=175, end_page=186)
ChapterInfo(title='Chapter 8: Program Design for Maximal Hypertrophy', start_page=187, end_page=219)
ChapterInfo(title='Chapter 9: Nutrition for Hypertrophy', start_page=220, end_page=239)


In [16]:
def extract_chapter(pdf_path, start_page, end_page, output_path):
    src = fitz.open(pdf_path)
    dst = fitz.open()

    # fitz uses zero-based indexing
    dst.insert_pdf(
        src,
        from_page=start_page - 1,
        to_page=end_page - 1,
    )

    dst.save(output_path)
    dst.close()
    src.close()

In [17]:
def extract_all_chapters(
    pdf_path: Path,
    chapters: list[ChapterInfo],
    chapters_dir: Path,
) -> list[Path]:
    
    chapter_paths = []

    for index, chapter in enumerate(chapters, start=1):

        output_path = chapters_dir / f"chapter_{index:02d}.pdf"

        extract_chapter(
            pdf_path=pdf_path,
            start_page=chapter.start_page,
            end_page=chapter.end_page,
            output_path=output_path,
        )
        chapter_paths.append(output_path)

    return chapter_paths



In [18]:
chapter_paths = extract_all_chapters(
    pdf_path=book_path,
    chapters=chapter_infos,
    chapters_dir=paths.chapters_dir,
)


In [19]:
from inspect import signature
from docling.document_converter import DocumentConverter

print(signature(DocumentConverter.convert))

(self, source: Union[pathlib.Path, str, docling_core.types.io.DocumentStream, docling.datamodel.base_models.HttpSource], headers: Optional[dict[str, str]] = None, raises_on_error: bool = True, max_num_pages: int = 9223372036854775807, max_file_size: int = 9223372036854775807, page_range: Annotated[Tuple[int, int], AfterValidator(func=<function _validate_page_range at 0x73e528c73600>)] = (1, 9223372036854775807)) -> docling.datamodel.document.ConversionResult


In [20]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

def create_converter() -> DocumentConverter:
    pipeline_options = PdfPipelineOptions()

    pipeline_options.do_ocr = False
    pipeline_options.do_table_structure = False
    pipeline_options.do_picture_description = False
    pipeline_options.do_picture_classification = False

    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options
            )
        }
    )




In [21]:
def convert_pdf(
    converter: DocumentConverter,
    pdf_path: Path,
):
    result = converter.convert(pdf_path)

    return result.document



In [ ]:
from collections import Counter

counter = Counter()

for item, _ in doc.iterate_items():
    counter[str(getattr(item, "label", "NONE"))] += 1

counter

In [ ]:
from inspect import signature

print(signature(doc.delete_items))

In [22]:
from copy import deepcopy
from typing import Iterable

ALLOWED_LABELS = {
    "section_header",
    "text",
    "list_item",
}


def keep_labels(doc, allowed_labels: Iterable[str]):
    """
    Return a copy of a DoclingDocument keeping only the specified labels.

    Parameters
    ----------
    doc : DoclingDocument
        Original document.

    allowed_labels : Iterable[str]
        Labels to keep.
        Example:
            {
                "section_header",
                "text",
                "list_item",
            }

    Returns
    -------
    DoclingDocument
        Filtered copy of the document.
    """

    doc = deepcopy(doc)

    allowed_labels = {label.lower() for label in allowed_labels}

    items_to_delete = []

    for item, _ in doc.iterate_items():
        label = str(getattr(item, "label", "")).lower()

        if label not in allowed_labels:
            items_to_delete.append(item)

    if items_to_delete:
        doc.delete_items(node_items=items_to_delete)

    return doc

In [23]:
def add_chapter_title(
    markdown: str,
    chapter_title: str,
) -> str:

    return f"# {chapter_title}\n\n{markdown}"

In [24]:
def convert_all_chapters_to_markdown(
    converter: DocumentConverter,
    chapter_infos: list[ChapterInfo],
    chapter_paths: list[Path],
    markdown_dir: Path,
) -> list[Path]:

    markdown_paths = []

    for chapter, chapter_path in zip(
        chapter_infos,
        chapter_paths,
    ):

        doc = convert_pdf(
            converter=converter,
            pdf_path=chapter_path,
        )

        doc = keep_labels(
            doc,
            ALLOWED_LABELS,
        )

        markdown = doc.export_to_markdown()

        markdown = add_chapter_title(
            markdown=markdown,
            chapter_title=chapter.title,
        )

        markdown_path = (
            markdown_dir /
            f"{chapter_path.stem}.md"
        )

        save_markdown(
            markdown=markdown,
            output_path=markdown_path,
        )

        markdown_paths.append(markdown_path)

    return markdown_paths

In [25]:
converter = create_converter()

markdown_paths = convert_all_chapters_to_markdown(
    converter=converter,
    chapter_infos=chapter_infos,
    chapter_paths=chapter_paths,
    markdown_dir=paths.markdown_dir,
)

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 9ed16645843556d8544034deb3a9c0db
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'nemotron-ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered layout engines: ['layout_object_detection', 'docling_layout_default', 'docling_experimental_table_crops_layout']
INFO:docling.utils.accelerator_utils:Accelerator device: 'cuda:0'
INFO:docl

In [26]:
import re

def normalize_title(title: str) -> str:

    title = re.sub(
        r"^Chapter\s+\d+\s*:\s*",
        "",
        title,
        flags=re.IGNORECASE,
    )

    return title.strip().casefold()


def remove_duplicate_chapter_title(
    markdown: str,
) -> str:

    lines = markdown.splitlines()

    if not lines:
        return markdown

    if not lines[0].startswith("# "):
        return markdown

    chapter_title = normalize_title(
        lines[0].removeprefix("# ")
    )

    for i in range(1, len(lines)):

        line = lines[i].strip()

        if not line:
            continue

        if line.startswith("## "):

            section_title = normalize_title(
                line.removeprefix("## ")
            )

            if section_title == chapter_title:
                del lines[i]

            break

        break

    return "\n".join(lines)

In [27]:

def load_markdown(markdown_path: Path) -> str:
    return markdown_path.read_text(encoding="utf-8")

def clean_markdown(markdown: str) -> str:

    markdown = markdown.strip()

    markdown = remove_duplicate_chapter_title(
        markdown,
    )

    # Remove figure/image IDs
    markdown = re.sub(
        r"E\d+/[^\n]+",
        "",
        markdown,
    )

    # Remove figure references
    markdown = re.sub(
        r"\b[Ff]igure\s+\d+(?:\.\d+)?\b",
        "",
        markdown,
    )

    # Remove table references
    markdown = re.sub(
        r"\b[Tt]able\s+\d+(?:\.\d+)?\b",
        "",
        markdown,
    )

    # Remove page labels (e.g. "1 chapter")
    markdown = re.sub(
        r"^\d+\s+chapter\s*$",
        "",
        markdown,
        flags=re.MULTILINE,
    )
    
    # Collapse multiple spaces
    markdown = re.sub(
        r"[ \t]{2,}",
        " ",
        markdown,
    )

    # Collapse multiple blank lines
    markdown = re.sub(
        r"\n{3,}",
        "\n\n",
        markdown,
    )

    markdown = re.sub(
        r"\s*\(continued\)",
        "",
        markdown,
        flags=re.IGNORECASE,
    )

    markdown = re.sub(
        r"(\w)-\n(\w)",
        r"\1\2",
        markdown,
    )

    markdown = re.sub(
        r"[ \t]+\n",
        "\n",
        markdown,
    )

    return markdown

def clean_all_markdown(
    markdown_paths: list[Path],
) -> None:

    for markdown_path in markdown_paths:

        markdown = load_markdown(markdown_path)

        markdown = clean_markdown(markdown)

        save_markdown(
            markdown,
            markdown_path,
        )

In [28]:
clean_all_markdown(markdown_paths)

In [29]:
from langchain_core.prompts import ChatPromptTemplate


ROUTING_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert in resistance training and exercise science.

Your task is to route a book chapter into exactly ONE collection based on its PRIMARY objective.

Available collections:

- principles
  Universal training concepts that apply regardless of the trainee's goal.

  Examples:
  - recovery science
  - progressive overload
  - periodization
  - rep range research
  - deload protocols
  - injury prevention

- hypertrophy
  Chapters primarily focused on maximizing skeletal muscle growth.

- strength
  Chapters primarily focused on maximizing force production and strength performance.

Routing rules:

1. Classify according to the chapter's PRIMARY objective, not every topic it mentions.

2. Ignore supporting material. Chapters about hypertrophy or strength often discuss recovery, anatomy, physiology, biomechanics, fatigue, and other general concepts to explain their main subject.

3. Choose "principles" ONLY when the chapter's main purpose is teaching concepts that are generally applicable across multiple training goals.

4. If the chapter's primary goal is muscle growth, choose "hypertrophy".

5. If the chapter's primary goal is strength development, choose "strength".

Return ONLY valid JSON matching this schema:

{{
    "collection": "principles | hypertrophy | strength"
}}
            """,
        ),
        (
            "human",
            "{chapter}",
        ),
    ]
)

In [30]:
from langchain_core.language_models import BaseChatModel

def route_chapter(
    markdown: str,
    llm: BaseChatModel,
) -> CollectionRoute:

    structured_llm = llm.with_structured_output(CollectionRoute, method="json_mode",)

    chain = ROUTING_PROMPT | structured_llm

    return chain.invoke(
        {
            "chapter": markdown,
        }
    )


In [31]:
def prepare_chapter_for_routing(markdown: str) -> str:

    chapter_title = None
    section_titles = []

    for line in markdown.splitlines():

        line = line.strip()

        if line.startswith("# "):
            chapter_title = line.removeprefix("# ").strip()

        elif line.startswith("## "):
            section_titles.append(
                line.removeprefix("## ").strip()
            )

    if chapter_title is None:
        raise ValueError(
            "No chapter title found."
        )

    result = [
        chapter_title,
        "",
        "Sections:",
    ]

    result.extend(
        f"- {section}"
        for section in section_titles
    )

    return "\n".join(result)

In [32]:
markdown_paths[0]

PosixPath('data/books/science_and_development_of_muscle_hypertrophy/markdown/chapter_01.md')

In [33]:
for i in range(9):
    markdown_path = markdown_paths[i]
    markdown = load_markdown(markdown_path)

    routing_input = prepare_chapter_for_routing(markdown)
    print(routing_input)
    print("#"*50)

Chapter 1: Hypertrophy-Related Responses and Adaptations to Exercise Stress

Sections:
- Neuromuscular System
- Structure and Function
- Sliding Filament Theory
- Motor Unit
- Fiber T ypes
- Responses and Adaptations
- KEY POINT
- Neural Drive
- Muscle Activation
- Motor Unit Synchronization
- Antagonist Coactivation
- Doublets
- Protein Balance
- KEY POINT
- Hypertrophy
- Parallel and In-Series (Serial) Hypertro-
- KEY POINT
- KEY POINT
- Hyperplasia
- Endocrine, Paracrine, and Autocrine Systems
- Responses and Adaptations of Hormones
- Insulin-Like Growth Factor 1
- Growth Hormone
- Testosterone
- Insulin
- Acute Versus Chronic Hormonal Responses
- KEY POINT
- Responses and Adaptations of Myokines
- Mechano Growth Factor
- Interleukins
- Myostatin
- Other Myokines
- KEY POINT
- TAKE-HOME POINTS
##################################################
Chapter 2: Mechanisms of Hypertrophy

Sections:
- Mechanical Tension
- Mechanotransduction
- KEY POINT
- Signaling Pathways
- KEY POINT
- PI3

In [34]:
from enum import Enum

class LLMProvider(Enum):
    GEMINI = "gemini"
    GROQ = "groq"
    NVIDIA = "nvidia"
    OPENROUTER = "openrouter"
    ITI = "iti"


@dataclass(frozen=True)
class LLMConfig:
    provider: LLMProvider
    model: str
    temperature: float = 0.0




In [35]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_openai import ChatOpenAI

def create_llm(config: LLMConfig) -> BaseChatModel:

    if config.provider == LLMProvider.GEMINI:
        return ChatGoogleGenerativeAI(
            model=config.model,
            temperature=config.temperature,
        )

    if config.provider == LLMProvider.GROQ:
        return ChatGroq(
            model=config.model,
            temperature=config.temperature,
        )
    
    if config.provider == LLMProvider.NVIDIA:
        return ChatNVIDIA(
            model=config.model,
            temperature=config.temperature,
        )
    
    if config.provider == LLMProvider.OPENROUTER:
        return ChatOpenAI(
            openai_api_base="https://openrouter.ai/api/v1",
            model=config.model,
            api_key=os.getenv("OPENROUTER_API_KEY"),
            temperature=config.temperature,
        )
    
    if config.provider == LLMProvider.ITI:
        return ChatOpenAI(
            base_url="http://apiaccess.iti.net.eg/api/v1",
            model=config.model,
            api_key=os.getenv("ITI_API_KEY"),
            temperature=config.temperature,
        )

    raise ValueError(
        f"Unsupported provider: {config.provider}"
    )

In [36]:

config = LLMConfig(
    provider=LLMProvider.GROQ,
    model="openai/gpt-oss-120b",
)

llm = create_llm(config)

In [37]:

def route_all_chapters(
    markdown_paths: list[Path],
    llm,
) -> dict[str, CollectionRoute]:

    routes = {}

    for markdown_path in markdown_paths:

        markdown = load_markdown(markdown_path)

        routing_input = prepare_chapter_for_routing(
            markdown
        )

        route = route_chapter(
            markdown=routing_input,
            llm=llm,
        )

        routes[markdown_path.name] = route

    return routes

In [38]:
chapter_routes = route_all_chapters(
    markdown_paths=markdown_paths,
    llm=llm,
)

chapter_routes

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


{'chapter_01.md': CollectionRoute(collection='hypertrophy'),
 'chapter_02.md': CollectionRoute(collection='hypertrophy'),
 'chapter_03.md': CollectionRoute(collection='hypertrophy'),
 'chapter_04.md': CollectionRoute(collection='hypertrophy'),
 'chapter_05.md': CollectionRoute(collection='hypertrophy'),
 'chapter_06.md': CollectionRoute(collection='hypertrophy'),
 'chapter_07.md': CollectionRoute(collection='hypertrophy'),
 'chapter_08.md': CollectionRoute(collection='hypertrophy'),
 'chapter_09.md': CollectionRoute(collection='hypertrophy')}

In [39]:
class Chunk(BaseModel):
    id: str
    
    text: str

    book: str
    chapter: str
    collection: Literal[
    "principles",
    "hypertrophy",
    "strength",
]

    chunk_index: int

    metadata: PrinciplesMetadata | GoalNamespaceMetadata | None = None


class ChunkingConfig(BaseModel):
    chunk_size: int = 800
    chunk_overlap: int = 150
    
    separators: list[str] = Field(
        default_factory=lambda: [
            "\n\n",
            "\n",
            " ",
            "",
        ]
    )



In [40]:
def extract_chapter_title(
    markdown: str,
) -> str:

    for line in markdown.splitlines():

        line = line.strip()

        if line.startswith("# "):
            return line.removeprefix("# ").strip()

    raise ValueError(
        "Chapter title not found."
    )

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_markdown(
    markdown: str,
    markdown_path: Path,
    chapter_routes: dict[str, CollectionRoute],
    book_info: BookInfo,
    config: ChunkingConfig,
    MIN_CHARS: int= 120
) -> list[Chunk]:

    chapter = extract_chapter_title(
        markdown,
    )

    collection = chapter_routes[
        markdown_path.name
    ].collection

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.chunk_size,
        chunk_overlap=config.chunk_overlap,
        separators=config.separators,
    )

    texts = splitter.split_text(
        markdown,
    )

    merged = []

    for text in texts:
        if len(text) < MIN_CHARS and merged:
            merged[-1] += "\n\n" + text
        else:
            merged.append(text)

    texts = merged


    return [
        Chunk(
            id="",
            text=text,
            book=book_info.title,
            chapter=chapter,
            collection=collection,
            chunk_index=i,
        )
        for i, text in enumerate(texts)
    ]

In [42]:
markdown = load_markdown(
    markdown_paths[0]
)
chunking_config = ChunkingConfig()

chunks = chunk_markdown(
    markdown=markdown,
    markdown_path=markdown_paths[0],
    chapter_routes=chapter_routes,
    book_info=book,
    config=chunking_config,
    MIN_CHARS=120
)

len(chunks)

171

In [43]:
chunks[0]

Chunk(id='', text='# Chapter 1: Hypertrophy-Related Responses and Adaptations to Exercise Stress\n\nTo comprehend the many factors related to maximizing skeletal muscle hypertrophy, it is essential to have a foundational knowledge of how the body reacts and adapts to exercise stress. This chapter reviews the structure and function of the neuromuscular system and the responses and adaptations of the neuromuscular, endocrine, paracrine, and autocrine systems. Although these systems are discussed separately, they are integrally connected; their interactions ultimately mediate lean tissue growth.\n\n## Neuromuscular System', book='Science and Development of Muscle Hypertrophy', chapter='Chapter 1: Hypertrophy-Related Responses and Adaptations to Exercise Stress', collection='hypertrophy', chunk_index=0, metadata=None)

In [44]:
def chunk_all_chapters(
    markdown_paths: list[Path],
    chapter_routes: dict[str, CollectionRoute],
    book_info: BookInfo,
    config: ChunkingConfig,
) -> list[Chunk]:

    chunks = []

    for markdown_path in markdown_paths:

        markdown = load_markdown(
            markdown_path,
        )

        chunks.extend(
            chunk_markdown(
                markdown=markdown,
                markdown_path=markdown_path,
                chapter_routes=chapter_routes,
                book_info=book_info,
                config=config,
            )
        )
    
    for i, chunk in enumerate(chunks):
        chunk.id = f"{book_info.slug}_{i:06d}"

    return chunks

In [45]:
all_chunks = chunk_all_chapters(
    markdown_paths=markdown_paths,
    chapter_routes=chapter_routes,
    book_info=book,
    config=chunking_config,
)

len(all_chunks)

1221

In [46]:
for chunk in all_chunks[:50]:
    print(len(chunk.text))
    print(len(chunk.text.split()))
    print(chunk.text)
    print("#"*50)

603
83
# Chapter 1: Hypertrophy-Related Responses and Adaptations to Exercise Stress

To comprehend the many factors related to maximizing skeletal muscle hypertrophy, it is essential to have a foundational knowledge of how the body reacts and adapts to exercise stress. This chapter reviews the structure and function of the neuromuscular system and the responses and adaptations of the neuromuscular, endocrine, paracrine, and autocrine systems. Although these systems are discussed separately, they are integrally connected; their interactions ultimately mediate lean tissue growth.

## Neuromuscular System
##################################################
593
89
## Neuromuscular System

A detailed discussion of the complexities of muscle hypertrophy requires a fundamental understanding of the neuromuscular systemin particular, the interaction between nerves and muscles that produces force to carry out human movement. Although a thorough exploration of the topic is beyond the scope of thi

In [47]:
all_chunks[67]

Chunk(id='science_and_development_of_muscle_hypertrophy_000067', text="The satellite cell response to a bout of resistance exercise lasts for many days, with effects peaking approximately 72 to 96 hours post-workout (23). The majority of evidence indicates that Type I fibers possess a greater resting number of satellite cells compared to Type II fibers, but it appears their population is increased to a greater extent in Type II fibers after resistance training (23). See .\n\nIt has been theorized that the most important hypertrophic role of satellite cells is their ability to retain a muscle's mitotic capacity by donating nuclei to existing myofibers (see ), thereby increasing the muscle's capacity to synthesize new contractile proteins (22, 144). This phenomenon is generally considered obligatory for maximizing overload-induced hypertrophy (60).", book='Science and Development of Muscle Hypertrophy', chapter='Chapter 1: Hypertrophy-Related Responses and Adaptations to Exercise Stress'

In [48]:
all_chunks[-1]

Chunk(id='science_and_development_of_muscle_hypertrophy_001220', text='- Nutrient timing around the exercise bout should be considered in the context of the peri-workout period. It seems prudent to consume high-quality protein (at a dose of approximately 0.4 to 0.5 g/kg of lean body mass) both pre- and post-exercise within about 4 to 6 hours of each other, depending on meal size. Those who train partially or fully fasted should consume protein as quickly as possible post-workout.', book='Science and Development of Muscle Hypertrophy', chapter='Chapter 9: Nutrition for Hypertrophy', collection='hypertrophy', chunk_index=136, metadata=None)

In [49]:
PRINCIPLES_METADATA_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert in exercise science.

Your task is to extract structured metadata for a text chunk from the "principles" knowledge namespace.

Guidelines:

1. Read the chunk carefully and identify its PRIMARY concepts.

2. Every selected value must be explicitly supported by the chunk.

3. Never select a label simply because it is related to the topic or because the field cannot be empty.

4. Choose only labels that represent major concepts of the chunk, not passing mentions or examples.

5. Multiple values are allowed only when they are equally central to the chunk.

6. Prefer precision over completeness. It is better to return fewer correct labels than many weakly supported ones.

7. Use only the provided schema and return only the structured output.
            """,
        ),
        (
            "human",
            """
Book:
{book}

Chapter:
{chapter}

Chunk:
{text}
            """,
        ),
    ]
)

In [50]:
GOAL_METADATA_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert in exercise science.

Your task is to extract structured metadata for a text chunk from a goal-specific knowledge namespace.

Guidelines:

1. Read the chunk carefully and identify its PRIMARY concepts.

2. Every selected value must be explicitly supported by the chunk.

3. Never select a label simply because it is related to the chapter, the book, or the collection.

4. Collection, book, and chapter titles are provided only as context to resolve ambiguity. The chunk itself is the ground truth.

5. Do not select labels based on passing mentions or examples.

6. Multiple values are allowed only when they are equally central to the chunk.

7. For fields such as muscle or experience level, use "all" only when the chunk genuinely applies broadly and does not focus on a specific subgroup.

8. Prefer precision over completeness. It is better to return fewer correct labels than many weakly supported ones.

9. Use only the provided schema and return only the structured output.
            """,
        ),
        (
            "human",
            """
Collection:
{collection}

Book:
{book}

Chapter:
{chapter}

Chunk:
{text}
            """,
        ),
    ]
)

In [51]:
import time


def generate_metadata(
    chunk: Chunk,
    llm: BaseChatModel,
) -> Chunk:

    if chunk.collection == "principles":

        prompt = PRINCIPLES_METADATA_PROMPT
        schema = PrinciplesMetadata

    else:

        prompt = GOAL_METADATA_PROMPT
        schema = GoalNamespaceMetadata

    structured_llm = llm.with_structured_output(
        schema,
    )

    chain = prompt | structured_llm

    while True:

        try:

            metadata = chain.invoke(
                {
                    "collection": chunk.collection,
                    "book": chunk.book,
                    "chapter": chunk.chapter,
                    "text": chunk.text,
                }
            )

            break

        except Exception as e:
            error_msg = str(e).lower()
            
            if (
                "429" in error_msg 
                or "503" in error_msg
                or "rate_limit" in error_msg 
                or "resourceexhausted" in error_msg
                or "resource_exhausted" in error_msg
            ):
                print(f"wait 60 sec")
                time.sleep(60)
                continue  
            else:
                raise e
            

    return chunk.model_copy(
    update={
        "metadata": metadata,
    }
)

In [52]:
import asyncio
import time

async def generate_metadata_async(chunk: Chunk, llm: BaseChatModel) -> Chunk:
    if chunk.collection == "principles":
        prompt = PRINCIPLES_METADATA_PROMPT
        schema = PrinciplesMetadata
    else:
        prompt = GOAL_METADATA_PROMPT
        schema = GoalNamespaceMetadata

    structured_llm = llm.with_structured_output(schema)
    chain = prompt | structured_llm

    while True:
        try:
            metadata = await chain.ainvoke(
                {
                    "collection": chunk.collection,
                    "book": chunk.book,
                    "chapter": chunk.chapter,
                    "text": chunk.text,
                }
            )
            break

        except Exception as e:
            error_msg = str(e).lower()
            if (
                "429" in error_msg 
                or "503" in error_msg
                or "rate_limit" in error_msg 
                or "resource" in error_msg
                or "timeout" in error_msg
                or "time out" in error_msg
            ):
                print(f"[{chunk.id}] wait 60 sec")
                await asyncio.sleep(60)
                continue
            else:
                raise e

    return chunk.model_copy(update={"metadata": metadata})

In [53]:
import json
def save_chunk(
    chunk: Chunk,
    chunks_dir: Path,
) -> None:

    path = chunks_dir / f"{chunk.id}.json"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            chunk.model_dump(),
            f,
            ensure_ascii=False,
            indent=2,
        )

def load_chunk(
    path: Path,
) -> Chunk:

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return Chunk.model_validate(data)

def chunk_exists(
    chunk: Chunk,
    chunks_dir: Path,
) -> bool:

    return (chunks_dir / f"{chunk.id}.json").exists()



def generate_metadata_for_all_chunks(
    chunks: list[Chunk],
    llm: BaseChatModel,
    chunks_dir: Path,
) -> list[Chunk]:

    chunks_with_metadata = []

    for i,chunk in enumerate(chunks):

        # Resume
        if chunk_exists(chunk, chunks_dir):
            print(f"Skipping {chunk.id}")
            continue

        chunk = generate_metadata(
            chunk=chunk,
            llm=llm,
        )

        save_chunk(
            chunk=chunk,
            chunks_dir=chunks_dir,
        )

        chunks_with_metadata.append(chunk)
        if i % 5 == 0 :
            print(f"{i + 1} chunks complected")

        time.sleep(0.4)

    return chunks_with_metadata

In [54]:
import asyncio

async def generate_metadata_for_all_chunks_async(
    chunks: list[Chunk],
    llm: BaseChatModel,
    chunks_dir: Path,
    max_concurrent_tasks: int = 10, 
) -> list[Chunk]:

    chunks_with_metadata = []
    tasks = []
    skipped_count = 0

    semaphore = asyncio.Semaphore(max_concurrent_tasks)

    for chunk in chunks:
        if chunk_exists(chunk, chunks_dir):
            existing_chunk = load_chunk(chunks_dir / f"{chunk.id}.json")

            if existing_chunk.metadata is not None:
                chunks_with_metadata.append(existing_chunk)
                skipped_count += 1
                continue
        
        tasks.append(generate_metadata_with_save_async(chunk, llm, chunks_dir, semaphore))

    if skipped_count > 0:
        print(f"Skipping {skipped_count} already processed chunks. ⏩")

    if not tasks:
        print("All chunks are already processed! 🎉")
        return chunks_with_metadata

    print(f"Starting controlled parallel processing for {len(tasks)} chunks (Max {max_concurrent_tasks} at a time)... 🚀")

    processed_chunks = await asyncio.gather(*tasks)

    chunks_with_metadata.extend(processed_chunks)
    
    print(f"Done! All {len(chunks_with_metadata)} chunks are now ready. ✅")
    return chunks_with_metadata


async def generate_metadata_with_save_async(
    chunk: Chunk, 
    llm: BaseChatModel, 
    chunks_dir: Path,
    semaphore: asyncio.Semaphore
) -> Chunk:
    
    async with semaphore:
        updated_chunk = await generate_metadata_async(chunk=chunk, llm=llm)
        
        save_chunk(chunk=updated_chunk, chunks_dir=chunks_dir)
        
        return updated_chunk

In [55]:
# config = LLMConfig(
#     provider=LLMProvider.NVIDIA,
#     model="nvidia/nemotron-3-super-120b-a12b",
# )

config = LLMConfig(
    provider=LLMProvider.GROQ,
    model="openai/gpt-oss-120b",
)

# config = LLMConfig(
#    provider=LLMProvider.GEMINI,
#    model="models/gemini-2.5-flash"
# )

# config = LLMConfig(
#     provider=LLMProvider.OPENROUTER,
#     model="nvidia/nemotron-3-ultra-550b-a55b:free",
#     temperature=0
# )

# config = LLMConfig(
#     provider=LLMProvider.ITI,
#     model="anthropic.claude-sonnet-4-6",
#     temperature=0
# )

llm = create_llm(config)

In [56]:
logging.disable(logging.CRITICAL)

In [57]:
chunks_metadata = await generate_metadata_for_all_chunks_async(
    chunks=all_chunks,
    llm=llm,
    chunks_dir=paths.chunks_dir,
    max_concurrent_tasks=5, 
)

Skipping 1219 already processed chunks. ⏩
Starting controlled parallel processing for 2 chunks (Max 5 at a time)... 🚀
Done! All 1221 chunks are now ready. ✅


In [58]:
import json
from pathlib import Path

broken_chunks = []

for path in paths.chunks_dir.glob("*.json"):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if data.get("metadata") is None:
        broken_chunks.append(path)

print(f"Found {len(broken_chunks)} chunks with metadata=None")

for path in broken_chunks[:10]:
    print(path.name)

Found 0 chunks with metadata=None
